# **Perceptron Simple**
## Materia: Fundamentos de Redes Neuronales
## Grupo: 4
## Integrantes: Juan Gonzalez, Facundo y Matias Siles
## Fecha: 28/04/2026

#Marco Teorico

*   Ajuste de pesos:
$$
\Delta w = \eta(\zeta^\mu - O^\mu)\xi_i^\mu
$$

*   Salida obtenida
$$
O = \theta(\sum_{i=1}^n w_i\xi_i - b)
$$

*   Funcion signo
$$
\theta(x) = \begin{cases}1 \quad x\ge0 \\ 0 \end{cases}
$$

*   Funcion Logistica(Sigmoidea)

$$
\theta(x) = \frac{1}{1 + e^{-2\beta h}}
$$

# Utils

In [ ]:
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Funciones auxiliares

def calcular_activacion(h, tipo, beta=1):
  """Funcion signo y logistica"""
  if tipo == "signo":
    return 1 if h >= 0 else -1

  elif tipo == "sigmoidea":
    return 1 / (1 + np.exp(-2*beta*h))

  else:
    raise ValueError("Tipo de Activacion Invalida")

def derivada_sigmoidea(O, beta=1):
  return 2 * beta * O * (1 - O)

def calcular_error(X, y, w, tipo_error, tipo_activacion):
  error = 0
  for i_X in range(len(X)):
    O = calcular_activacion(X[i_X] @ w, tipo_activacion)

    if tipo_error == "lineal":
      error += abs(y[i_X] - O)

    elif tipo_error == "cuadratico":
      error += 0.5 * abs(y[i_X] - O)**2

    else:
      raise ValueError("Tipo de Error Invalido")

  return error

def mostrar_resultados(i, error_min, w_min):
  print(f"i: {i}")
  print(f"Min. error: {np.round(error_min,2 )}")
  print(f"Mejores pesos: {w_min}")

# Perceptron Simple Lineal

In [ ]:
def perceptron_lineal(X, y, eta, cota):
  """Entrenamiento"""

  p, n = X.shape
  i = 0
  w = np.zeros(n)
  w[-1] = 1 # bias
  error_min = p * 2
  error = 1

  while error > 0 and i < cota:
    i_X = np.random.randint(0, p)

    h = X[i_X] @ w
    O = calcular_activacion(h, tipo="signo")

    delta_w = eta * (y[i_X] - O) * X[i_X]
    w += delta_w

    error = calcular_error(X, y, w, tipo_error="lineal", tipo_activacion="signo")

    if error < error_min:
      error_min = error
      w_min = w.copy()

    i += 1

  return i, error_min, w_min

# Perceptron Simple No Lineal

In [ ]:
def perceptron_nolineal(X, y, eta, cota):
  """Entrenamiento"""

  p, n = X.shape
  i = 0
  w = np.zeros(n)
  w[-1] = 1 # bias
  error_min = p * 2
  error = 1

  while error > 0 and i < cota:
    i_X = np.random.randint(0, p)

    h = X[i_X] @ w
    O = calcular_activacion(h, tipo="sigmoidea")
    grad_O = derivada_sigmoidea(O)

    delta_w = eta * (y[i_X] - O) * grad_O * X[i_X]
    w += delta_w

    error = calcular_error(X, y, w, tipo_error="cuadratico",
                           tipo_activacion="sigmoidea")

    if error < error_min:
      error_min = error
      w_min = w.copy()

    i += 1

  return i, error_min, w_min

# Datos

In [ ]:
X = np.array([
    [-1,  1],
    [ 1, -1],
    [-1, -1],
    [ 1,  1]
])

y_lin = np.array([-1, -1, -1, 1])
y_non = np.array([1, 1, -1, -1])

In [ ]:
X_tp = np.loadtxt('/content/drive/MyDrive/datos_tp_perceptron/TP1-ej2-Conjunto-entrenamiento.txt')
y_tp = np.loadtxt('/content/drive/MyDrive/datos_tp_perceptron/TP1-ej2-Salida-deseada.txt').reshape(-1, 1) / 100 # redimensionar

In [ ]:
# Agregar bias
X = np.insert(X, 2, 1, axis=1)
X_tp = np.insert(X_tp, 3, 1, axis=1)

#**Resultados**

##1. Perceptron Simple con Funcion de Activacion Escalon para AND y XOR

In [ ]:
# AND
i_lin, error_min_lin, w_min_lin = perceptron_lineal(X, y_lin, eta=0.1, cota=100)
mostrar_resultados(i_lin, error_min_lin, w_min_lin)

i: 22
Min. error: 0
Mejores pesos: [ 0.4  0.4 -0.2]


In [ ]:
# XOR
i_non, error_min_non, w_min_non = perceptron_lineal(X, y_non, eta=0.1, cota=100)
mostrar_resultados(i_non, error_min_non, w_min_non)

i: 100
Min. error: 2
Mejores pesos: [-0.4 -0.4  0.6]


*   Conclusion: Podemos decir que los problemas qu resuelve un perceptron simple con funcion escalon son limitados con respecto a los diferentes tipos de problemas. Pudiendo solo aprender problemas linealmente separables(AND)  

##2. Perceptron Simple Lineal y No Lineal en los Datos del TP

###Evaluacion de ambos perceptrones

In [ ]:
# Lineal
i_tp, error_min_tp, w_min_tp = perceptron_lineal(X_tp, y_tp, eta=0.1, cota=700)
mostrar_resultados(i_tp, error_min_tp, w_min_tp)

i: 700
Min. error: [118.68]
Mejores pesos: [ 0.01067415  0.05880674 -0.1117906   0.7734559 ]


In [ ]:
# No lineal
i_tp, error_min_tp, w_min_tp = perceptron_nolineal(X_tp, y_tp, eta=0.1, cota=700)
mostrar_resultados(i_tp, error_min_tp, w_min_tp)

i: 700
Min. error: [0.]
Mejores pesos: [ 0.24885218  0.24894606  0.24816119 -0.24455002]


*   Conclusion: Durante el aprendizaje de ambos perceptrones, el perceptron lineal no es capaz de aprender con exito, dando un error elevado. Por otro lado, el perceptron lineal muestra aprendizaje pero ajustando la cantidad de iteraciones a una cantidad elevada cerca de 1000 iteraciones. Esto significa que los datos presentan un problema no linealmente separable

*   Grafico Perceptron No Lineal: https://www.desmos.com/3d/nnk1k2lbnt?lang=es

###Evaluacion del Perceptron No Lineal en subconjuntos train y test

Datos

In [ ]:
idx = np.random.permutation(len(X_tp)) # Indices unicos aleatorios

split = int(0.8 * len(X_tp))

# Dividir idx en train y test
train_idx = idx[:split]
test_idx = idx[split:]

# Dividir en subconjuntos segun los indices aleatorios
X_tp_train = X_tp[train_idx]
X_tp_test = X_tp[test_idx]

y_tp_train = y_tp[train_idx]
y_tp_test = y_tp[test_idx]

Entrenamiento

In [ ]:
i_non, error_min_non, w_min_non = perceptron_nolineal(X_tp_train, y_tp_train, eta=0.1, cota=700)
mostrar_resultados(i_non, error_min_non, w_min_non)

i: 700
Min. error: [0.]
Mejores pesos: [ 0.25367538  0.25361444  0.2528892  -0.24932367]


Test

In [ ]:
i_non_test, error_min_non_test, w_min_non_test = perceptron_nolineal(X_tp_test, y_tp_test, eta=0.1, cota=700)
mostrar_resultados(i_non_test, error_min_non_test, w_min_non_test)

i: 700
Min. error: [0.]
Mejores pesos: [ 0.24410101  0.24453423  0.24502896 -0.23521054]


*   Conclusion: El modelo se entreno con exito y generalizo bien con los datos test, dando un error 0 en ambos subconjuntos con una iteracion alta aproximada a 1000 iteraciones

*   Sugerencias: En este caso los subconjuntos train y test se eligieron de forma al azar en los datos de X, evitando sesgos y demas. Pero no es la mejor opcion si se busca el mejor conjunto de entrenamiento. Existen tecnicas como cross-validation que buscan entrenar y testear con todas las combinaciones posibles que se puede dividir en subconjuntos los datos, esta tecnica es mas confiable ya que todos los datos son evaluados y depende menos del azar. La tecnica cross-validation permite que el modelo pueda generalizar a un nivel mas alto con datos que nunca vio, permitiendo aprender mejor con el conjunto de entrenamiento y hacer predicciones mas precisas con datos nuevos.

# Glosario


*   p = Cantidad de entradas
*   n = Cantidad de dimensiones
*   COTA = Max. de iteraciones
*   i_x = Numero al azar entre 1 y p
*   y = Vector de la salida esperada
*   x = Array de vectores sobre el conjunto de entrenamiento
*   w = Vector de pesos
*   h = Excitacion
*   O = La salida de una entrada al azar en el perceptron
*   eta = Tasa de aprendizaje
*   pos = Posicion
*   Error = 1, Error inicial para entrenar el perceptron
*   arr = array
*   O_grad = Gradiente de la activacion
*   error cuadratico = eleva al cuadrado las diferencias para penalizar los errores mas grandes


#Referencias

Link: https://campus2026.unahur.edu.ar/pluginfile.php/417868/mod_resource/content/1/Perceptron.pdf